# 🕵️ Aula 6 — Testando Positivo (parte 2): Probabilidade Total e Teorema de Bayes

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Ele é a segunda parte do projeto que atravessa as **Aulas 5 a 8**: em cada aula atacamos o **mesmo problema** (o detector de mentiras da atividade em sala) com uma ferramenta nova. Na Aula 5 descobrimos, **simulando** 100.000 funcionários, que só **cerca de 31%** dos que falham no detector são realmente ladrões. Aqui, na Aula 6, as ferramentas são o **Teorema da Probabilidade Total** e o **Teorema de Bayes**, e com elas vamos chegar a esse mesmo número **só com contas**.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **A simulação** (quando houver): o computador repete a situação milhares de vezes;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo faz três coisas:

- **Carrega o `numpy`**, uma "calculadora" do Python que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios.
- **Carrega o `matplotlib`**, que desenha os gráficos.
- **Cria o nosso sorteador** (`rng`). É ele que vai "decidir" qual urna é escolhida, qual bola sai e onde está o prêmio no jogo da seção 4. O número **1** é a **semente**: faz o sorteio sair **sempre igual**, toda vez que alguém executa o notebook. Por isso os números na sua tela serão os mesmos citados nos textos. (Qualquer outro número serviria; 1 é só uma escolha.)

**O que é simular?** É usar o computador para repetir um experimento aleatório milhares de vezes, como se estivéssemos fazendo de verdade. Chamamos de:
- **empírico**: o que sai da simulação (o que "aconteceu");
- **teórico**: o que a conta da aula diz que deveria acontecer.

Se a conta estiver certa, os dois devem ficar **bem próximos**.

In [ ]:
import numpy as np                  # calculadora de números e sorteios
import matplotlib.pyplot as plt     # desenha gráficos

rng = np.random.default_rng(1)      # o sorteador, com semente 1 (resultados sempre iguais)
print("Tudo pronto!")

## 1. Quantos funcionários falham no detector?

### A situação
É a mesma da Aula 5. Numa grande loja, **10% dos funcionários são ladrões**. Todos passam por um detector de mentiras que acerta **80% das vezes**, para os dois tipos de pessoa:
- se a pessoa **é ladra**, o detector a **flagra** (ela "falha no teste") com probabilidade 0,8;
- se a pessoa **é honesta**, o detector erra e a **flagra por engano** com probabilidade 0,2.

### A pergunta
Escolhendo um funcionário qualquer, **qual a chance de ele falhar no teste?** Em símbolos: quanto vale $P(\text{falhou})$?

### A ferramenta da aula
**Teorema da Probabilidade Total.** Não sabemos direto a chance de falhar, porque ela **depende de quem a pessoa é**. Mas todo funcionário cai em exatamente um de dois grupos (ladrão ou honesto), e em cada grupo sabemos a chance de falhar. Então somamos as duas "rotas" até "falhou", cada uma com o peso do seu grupo:

$$P(\text{falhou}) = P(\text{falhou} \mid \text{ladrão})\,P(\text{ladrão}) + P(\text{falhou} \mid \text{honesto})\,P(\text{honesto})$$

### A resposta pela conta
$$P(\text{falhou}) = 0{,}8 \times 0{,}1 + 0{,}2 \times 0{,}9 = 0{,}08 + 0{,}18 = 0{,}26$$

Nesta seção não há sorteio: o código abaixo só **guarda os dados do problema** em variáveis (nomes fáceis de ler) e **faz a mesma conta** da fórmula.

In [ ]:
# os dados do problema
p_ladrao = 0.10
p_honesto = 1 - p_ladrao            # 0.90

p_falhar_dado_ladrao = 0.8          # o detector flagra corretamente o ladrão
p_falhar_dado_honesto = 0.2         # o detector erra e flagra o honesto

# Probabilidade Total: soma das duas rotas até "falhou"
p_falhou = p_falhar_dado_ladrao * p_ladrao + p_falhar_dado_honesto * p_honesto

print(f"P(falhou) = {p_falhar_dado_ladrao} × {p_ladrao} + {p_falhar_dado_honesto} × {p_honesto} = {p_falhou:.2f}")

**Como ler:** a conta dá **0,26**, ou seja, **26%** dos funcionários falham no teste. Na Aula 5, a simulação de 100.000 funcionários deu **26,0%** (26.007 pessoas). A conta confirma o que a simulação tinha mostrado, sem precisar sortear ninguém.

Repare de onde vêm esses 26%: **8 pontos** vêm de ladrões flagrados ($0{,}8 \times 0{,}1$) e **18 pontos** vêm de honestos flagrados por engano ($0{,}2 \times 0{,}9$). Guarde essa divisão: ela é a chave da próxima seção.

## 2. Quem falha é ladrão? (Bayes)

### A situação
A mesma loja. Agora sabemos que um funcionário **falhou** no teste.

### A pergunta
É a pergunta dos gerentes, que vão demitir quem falhar: **qual a chance de essa pessoa ser realmente ladra?** Em símbolos: $P(\text{ladrão} \mid \text{falhou})$.

### A ferramenta da aula
**Teorema de Bayes.** Nós conhecemos a probabilidade "no sentido do detector" ($P(\text{falhou} \mid \text{ladrão}) = 0{,}8$), mas queremos a do **sentido contrário**. Bayes faz essa inversão: é a fração dos que falharam que chegou lá **pela rota "ladrão"**:

$$P(\text{ladrão} \mid \text{falhou}) = \frac{P(\text{falhou} \mid \text{ladrão})\,P(\text{ladrão})}{P(\text{falhou})}$$

O denominador é exatamente o $P(\text{falhou})$ que a Probabilidade Total calculou na seção 1.

### A resposta pela conta
$$P(\text{ladrão} \mid \text{falhou}) = \frac{0{,}8 \times 0{,}1}{0{,}26} = \frac{0{,}08}{0{,}26} \approx 0{,}308$$

O código abaixo faz essa conta usando as variáveis da seção 1 (por isso é preciso ter executado aquela célula antes).

In [ ]:
# Bayes: a rota "ladrão" dividida pelo total de quem falhou
p_ladrao_dado_falhou = (p_falhar_dado_ladrao * p_ladrao) / p_falhou

print(f"P(ladrão | falhou) = ({p_falhar_dado_ladrao} × {p_ladrao}) / {p_falhou:.2f} = {p_ladrao_dado_falhou:.3f}")
print(f"Em porcentagem: {p_ladrao_dado_falhou:.1%}")

**Como ler:** a conta dá **0,308**, ou seja, **cerca de 31%**. É o mesmo número que a simulação da Aula 5 encontrou (0,305), agora obtido **só com a fórmula**.

**Por que tão baixo, se o detector "acerta 80%"?** Volte à divisão da seção 1: dos 26 pontos de quem falha, só **8** são ladrões e **18** são honestos. Como os honestos são muito mais numerosos (90% da loja), os erros do detector com eles pesam mais que os acertos com os ladrões.

**A lição da aula:** $P(\text{falhou} \mid \text{ladrão}) = 0{,}8$ e $P(\text{ladrão} \mid \text{falhou}) \approx 0{,}31$ são perguntas **diferentes**. O Teorema de Bayes é justamente a ferramenta que leva de uma à outra.

## 3. As duas urnas da aula

### A situação
É o exemplo dos slides. A **urna 1** tem **3 bolas brancas e 2 amarelas**; a **urna 2** tem **4 brancas e 2 amarelas**. Escolhe-se uma urna ao acaso (50% cada) e dela retira-se uma bola.

### A pergunta
(a) Qual a probabilidade de a bola ser **branca**?
(b) Se a bola saiu **branca**, qual a probabilidade de ter vindo da **urna 1**?

### A ferramenta da aula
É o mesmo esquema do detector, trocando "ladrão/honesto" por "urna 1/urna 2" e "falhou" por "branca":
- para (a), **Probabilidade Total**: a chance de branca depende da urna, então somamos as duas rotas;
- para (b), **Bayes**: invertemos a pergunta, da bola para a urna.

### A resposta pela conta
(a) $P(\text{branca}) = \frac{3}{5} \times 0{,}5 + \frac{4}{6} \times 0{,}5 = 0{,}300 + 0{,}333 \approx 0{,}633$

(b) $P(\text{urna 1} \mid \text{branca}) = \dfrac{\frac{3}{5} \times 0{,}5}{0{,}633} = \dfrac{0{,}300}{0{,}633} \approx 0{,}474$

A célula abaixo faz as duas contas no computador.

In [ ]:
p_urna1 = 0.5
p_urna2 = 0.5
p_branca_dado_urna1 = 3 / 5     # 3 brancas em 5 bolas
p_branca_dado_urna2 = 4 / 6     # 4 brancas em 6 bolas

# (a) Probabilidade Total: soma das duas rotas até "branca"
p_branca = p_branca_dado_urna1 * p_urna1 + p_branca_dado_urna2 * p_urna2
print(f"P(branca) = {p_branca_dado_urna1:.3f} × {p_urna1} + {p_branca_dado_urna2:.3f} × {p_urna2} = {p_branca:.3f}")

# (b) Bayes: de quem saiu branca, que fração veio pela rota "urna 1"
p_urna1_dado_branca = (p_branca_dado_urna1 * p_urna1) / p_branca
print(f"P(urna 1 | branca) = ({p_branca_dado_urna1:.3f} × {p_urna1}) / {p_branca:.3f} = {p_urna1_dado_branca:.3f}")

### A simulação
Agora o computador faz o experimento **300.000 vezes**. Em cada repetição, o sorteador faz duas coisas, na ordem:
1. escolhe a **urna** (1 ou 2, com 50% de chance cada);
2. tira uma bola **dessa urna**: sai branca com chance 3/5 se for a urna 1, e com chance 4/6 se for a urna 2.

No fim, o código **conta** em quantas repetições saiu branca (pergunta a). Para a pergunta (b), ele faz o mesmo "filtro" da Aula 5: **fica só com as repetições em que saiu branca** e conta quantas vieram da urna 1.

In [ ]:
n = 300_000

# Passo 1: em cada repetição, escolhe a urna (1 ou 2, 50% cada)
urna_escolhida = rng.integers(1, 3, size=n)

# Passo 2: tira a bola da urna escolhida (True = branca, False = amarela)
eh_branca = np.where(
    urna_escolhida == 1,
    rng.random(n) < p_branca_dado_urna1,     # se foi a urna 1: branca com chance 3/5
    rng.random(n) < p_branca_dado_urna2,     # se foi a urna 2: branca com chance 4/6
)

print(f"(a) P(branca) empírico: {eh_branca.mean():.3f}")
print(f"    P(branca) teórico:  {p_branca:.3f}")

# (b) filtro "dado que saiu branca": só as repetições em que a bola foi branca
urna_das_brancas = urna_escolhida[eh_branca]
print(f"(b) P(urna 1 | branca) empírico: {(urna_das_brancas == 1).mean():.3f}")
print(f"    P(urna 1 | branca) teórico:  {p_urna1_dado_branca:.3f}")

# gráfico: de onde vieram as bolas brancas
plt.bar(["Urna 1", "Urna 2"], [(urna_das_brancas == 1).sum(), (urna_das_brancas == 2).sum()], color=["goldenrod", "steelblue"])
plt.title("De qual urna vieram as bolas brancas?")
plt.ylabel("Número de bolas brancas")
plt.show()

**Como ler:**
- (a) a bola saiu branca em **0,633** das repetições, igual à conta (0,633).
- (b) entre as bolas brancas, **0,474** vieram da urna 1, também igual à conta (0,474). No gráfico, a barra da urna 2 é um pouco maior.

**Por que a urna 2 "ganha", se a escolha da urna foi 50%/50%?** Porque a urna 2 tem uma proporção de brancas um pouco maior (4/6 ≈ 0,67 contra 3/5 = 0,6). Ver uma bola branca é uma pequena **pista** a favor da urna 2, e Bayes mede exatamente o tamanho dessa pista: a chance da urna 1 cai de **50%** (antes de ver a bola) para **47,4%** (depois de ver que é branca).

## 4. Bônus de curiosidade: o problema de Monty Hall

Esta seção **não é obrigatória** para a disciplina; é uma curiosidade clássica de probabilidade, usada no vídeo recomendado desta aula como aplicação do Teorema de Bayes.

### A situação
Num programa de TV, há **3 portas**: atrás de uma há um prêmio, atrás das outras duas, nada. Você escolhe uma porta. O apresentador, **que sabe onde está o prêmio**, abre uma das **outras** portas, sempre uma **sem prêmio**, e pergunta se você quer **trocar** para a porta que sobrou.

### A pergunta
**Vale a pena trocar?** Qual a chance de ganhar ficando e qual a chance de ganhar trocando?

### A ferramenta da aula
O raciocínio da **Probabilidade Total**: separar os casos pela sua escolha inicial.
- Sua escolha inicial **estava certa** (chance 1/3): trocando, você **perde**.
- Sua escolha inicial **estava errada** (chance 2/3): o apresentador abre a outra porta vazia, e a porta que sobra **é a do prêmio**; trocando, você **ganha**.

### A resposta pela conta
$$P(\text{ganhar ficando}) = \tfrac{1}{3} \approx 0{,}333 \qquad P(\text{ganhar trocando}) = \tfrac{2}{3} \approx 0{,}667$$

### A simulação
O computador vai jogar **100.000 partidas**. Em cada uma, o sorteador escolhe a porta do prêmio e a porta que o jogador escolhe primeiro (0, 1 ou 2, ao acaso). Pelo raciocínio acima, não precisamos simular o apresentador abrindo a porta: **quem fica ganha quando acertou de primeira, e quem troca ganha exatamente quando errou de primeira.**

In [ ]:
n_jogos = 100_000

premio = rng.integers(0, 3, size=n_jogos)           # porta com o prêmio, sorteada em cada partida
escolha_inicial = rng.integers(0, 3, size=n_jogos)  # porta que o jogador escolhe primeiro

# quem fica com a escolha inicial ganha só quando ela já era a porta certa
ganha_ficando = (escolha_inicial == premio)

# quem troca ganha exatamente quando a escolha inicial NÃO era a certa ("~" = "não")
ganha_trocando = ~ganha_ficando

print(f"Vitórias ficando:  {ganha_ficando.sum()}  ({ganha_ficando.mean():.1%})   teórico: {1/3:.1%}")
print(f"Vitórias trocando: {ganha_trocando.sum()}  ({ganha_trocando.mean():.1%})   teórico: {2/3:.1%}")

# gráfico: vitórias de cada estratégia
plt.bar(["Ficar", "Trocar"], [ganha_ficando.sum(), ganha_trocando.sum()], color=["gray", "seagreen"])
plt.title("Monty Hall: 100.000 partidas")
plt.ylabel("Número de vitórias")
plt.show()

**Como ler:** ficando, o jogador ganhou **33,4%** das partidas (33.409); trocando, **66,6%** (66.591). Bem perto de 1/3 e 2/3. No gráfico, a barra de "Trocar" tem o dobro da altura: **trocar de porta dobra a chance de ganhar.**

É o mesmo tipo de surpresa do detector de mentiras: a **informação nova** (o apresentador, que sabe onde está o prêmio, abrir uma porta vazia) muda as probabilidades, mesmo que pareça que "não deveria fazer diferença".

## Para fechar

Neste notebook usamos as duas ideias da Aula 6:

- **Probabilidade Total:** quando a chance de um evento depende do "grupo" (ladrão/honesto, urna 1/urna 2), somamos as rotas de cada grupo, cada uma com o seu peso;
- **Teorema de Bayes:** para inverter a pergunta ("dado que falhou, é ladrão?"), pegamos a rota que interessa e dividimos pelo total da Probabilidade Total.

E guardamos um número: **26%** dos funcionários falham no detector. Na **Aula 7** vamos usar esse número para prever **quantos** funcionários, num grupo pequeno, devem falhar, tratando essa contagem como uma **variável aleatória discreta**.